# LJ-флюид и уравнение Ван-дер-Ваальса в Google Colab

Этот ноутбук является самодостаточной вычислительной работой: он моделирует Lennard-Jones-флюид в периодической 3D-коробке, строит EOS-таблицы, подбирает параметры уравнения Ван-дер-Ваальса и сохраняет графики сравнения. Локальные Python-модули проекта не импортируются.

## Физическая постановка

- Частицы взаимодействуют потенциалом Леннарда-Джонса в приведённых единицах.
- Система: периодическая кубическая 3D-коробка.
- Ансамбль: NVT, температура поддерживается Langevin-термостатом.
- Для каждой пары `(T, rho, seed)` считается одна EOS-точка.
- Траектории, координаты и морфологические классификации не сохраняются.
- Сохраняются только `data/eos_points.csv` и `data/eos_final_profiles.csv`.

## Установка зависимостей и проверка GPU

In [ ]:
# Colab: перед запуском выберите Runtime -> Change runtime type -> GPU.
!pip -q install openmm numpy pandas scipy matplotlib tqdm

import openmm as mm

OPENMM_PLATFORMS = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
CUDA_AVAILABLE = "CUDA" in OPENMM_PLATFORMS
print("OpenMM platforms:", OPENMM_PLATFORMS)
print("CUDA visible:", CUDA_AVAILABLE)
if not CUDA_AVAILABLE:
    print("WARNING: CUDA не видна. Расчёт пойдёт на CPU и может быть медленным. В Colab проверьте Runtime -> Change runtime type -> GPU.")


## Импорты

In [ ]:
from pathlib import Path
import json
import math
import shutil
from statistics import mean, stdev

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from tqdm.auto import tqdm

import openmm as mm
from openmm import app, unit


## PARAMS

Все параметры расчёта задаются здесь. Для большого расчёта в Colab обычно увеличивают `N`, `equil_steps`, `prod_steps`, а также списки `temperatures` и `densities`.

In [ ]:
PARAMS = {
    "N": 128,
    "sigma": 1.0,
    "epsilon": 1.0,
    "mass": 1.0,
    "rcut": 2.5,
    "dt": 0.002,
    "equil_steps": 500,
    "prod_steps": 1000,
    "sample_interval": 250,
    "friction": 0.2,
    "temperatures": [0.70, 0.90],
    "densities": [0.02, 0.06],
    "seeds": [1],
    "profile_bins": 40,
    "platform": "auto",      # auto -> CUDA -> OpenCL -> CPU -> Reference
    "gpu_precision": "mixed",
    "device_index": None,
}

DATA_DIR = Path("data")
FIGURES_DIR = Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
PARAMS


## Функции MD

In [ ]:
EOS_FIELDS = [
    "run_id", "T_target", "T_mean", "rho_target", "rho_mean", "N", "L",
    "dt", "equil_steps", "prod_steps", "seed", "P_mean", "P_std", "P_sem",
    "U_mean", "U_std", "U_sem", "K_mean", "E_mean", "status", "notes",
]
PROFILE_FIELDS = ["run_id", "bin", "z_min", "z_max", "z_center", "count"]

def available_platforms():
    return [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]

def select_platform(params):
    requested = str(params.get("platform", "auto"))
    names = available_platforms()
    if requested == "auto":
        for candidate in ("CUDA", "OpenCL", "CPU", "Reference"):
            if candidate in names:
                requested = candidate
                break
    elif requested not in names:
        raise RuntimeError(f"Requested OpenMM platform {requested!r} is unavailable. Available: {names}")
    platform = mm.Platform.getPlatformByName(requested)
    properties = {}
    if requested in {"CUDA", "OpenCL"}:
        properties["Precision"] = str(params.get("gpu_precision", "mixed"))
        if params.get("device_index") is not None:
            properties["DeviceIndex"] = str(params["device_index"])
    return platform, properties

def initialize_positions(N, L):
    n_side = math.ceil(N ** (1.0 / 3.0))
    spacing = L / n_side
    coords = []
    for ix in range(n_side):
        for iy in range(n_side):
            for iz in range(n_side):
                if len(coords) == N:
                    return np.asarray(coords, dtype=float)
                coords.append([(ix + 0.5) * spacing, (iy + 0.5) * spacing, (iz + 0.5) * spacing])
    return np.asarray(coords, dtype=float)

def create_lj_system(params, density):
    N = int(params["N"])
    sigma = float(params["sigma"])
    epsilon = float(params["epsilon"])
    mass = float(params["mass"])
    rcut = float(params["rcut"])
    L = float((N / density) ** (1.0 / 3.0))

    system = mm.System()
    for _ in range(N):
        system.addParticle(mass * unit.dalton)
    system.setDefaultPeriodicBoxVectors(
        unit.Quantity(mm.Vec3(L, 0, 0), unit.nanometer),
        unit.Quantity(mm.Vec3(0, L, 0), unit.nanometer),
        unit.Quantity(mm.Vec3(0, 0, L), unit.nanometer),
    )

    expr = "4*epsilon*((sigma/r)^12-(sigma/r)^6)-4*epsilon*((sigma/rcut)^12-(sigma/rcut)^6)"
    force = mm.CustomNonbondedForce(expr)
    force.addGlobalParameter("sigma", sigma * unit.nanometer)
    force.addGlobalParameter("epsilon", epsilon * unit.kilojoule_per_mole)
    force.addGlobalParameter("rcut", rcut * unit.nanometer)
    force.setNonbondedMethod(mm.CustomNonbondedForce.CutoffPeriodic)
    force.setCutoffDistance(rcut * unit.nanometer)
    force.setUseLongRangeCorrection(False)
    for _ in range(N):
        force.addParticle([])
    system.addForce(force)
    return system, L

def make_topology(N):
    topology = app.Topology()
    chain = topology.addChain()
    residue = topology.addResidue("LJ", chain)
    element = app.Element.getByAtomicNumber(18)
    for _ in range(N):
        topology.addAtom("Ar", element, residue)
    return topology

def get_positions(simulation):
    state = simulation.context.getState(getPositions=True)
    return np.asarray(state.getPositions(asNumpy=True).value_in_unit(unit.nanometer), dtype=float)

def lj_virial_pressure(positions, L, T, rcut=2.5, sigma=1.0, epsilon=1.0):
    V = L ** 3
    rho = len(positions) / V
    virial = 0.0
    rcut2 = rcut * rcut
    for i in range(len(positions) - 1):
        delta = positions[i + 1:] - positions[i]
        delta -= L * np.rint(delta / L)
        r2 = np.sum(delta * delta, axis=1)
        mask = (r2 > 0.0) & (r2 < rcut2)
        if not np.any(mask):
            continue
        inv_r2 = (sigma * sigma) / r2[mask]
        inv_r6 = inv_r2 ** 3
        inv_r12 = inv_r6 ** 2
        virial += float(np.sum(24.0 * epsilon * (2.0 * inv_r12 - inv_r6)))
    return float(rho * T + virial / (3.0 * V))

def compute_z_counts(positions, L, bins, run_id):
    counts, edges = np.histogram(positions[:, 2] % L, bins=bins, range=(0.0, L))
    rows = []
    for i, count in enumerate(counts):
        z_min = float(edges[i])
        z_max = float(edges[i + 1])
        rows.append({"run_id": run_id, "bin": i, "z_min": z_min, "z_max": z_max, "z_center": 0.5 * (z_min + z_max), "count": int(count)})
    return rows

def sample_stats(values):
    if len(values) == 0:
        return math.nan, math.nan, math.nan
    if len(values) == 1:
        return float(values[0]), 0.0, 0.0
    s = stdev(values)
    return float(mean(values)), float(s), float(s / math.sqrt(len(values)))

def run_eos_point(params, T, rho, seed, run_id):
    N = int(params["N"])
    epsilon = float(params["epsilon"])
    sigma = float(params["sigma"])
    rcut = float(params["rcut"])
    dt = float(params["dt"])
    equil_steps = int(params["equil_steps"])
    prod_steps = int(params["prod_steps"])
    sample_interval = int(params["sample_interval"])

    system, L = create_lj_system(params, rho)
    gas_r = unit.MOLAR_GAS_CONSTANT_R.value_in_unit(unit.kilojoule_per_mole / unit.kelvin)
    openmm_T = (T * epsilon / gas_r) * unit.kelvin
    integrator = mm.LangevinMiddleIntegrator(openmm_T, float(params["friction"]) / unit.picosecond, dt * unit.picoseconds)
    integrator.setRandomNumberSeed(int(seed))
    platform, properties = select_platform(params)
    simulation = app.Simulation(make_topology(N), system, integrator, platform, properties)
    simulation.context.setPositions(initialize_positions(N, L) * unit.nanometer)
    simulation.context.setVelocitiesToTemperature(openmm_T, int(seed))

    if equil_steps:
        simulation.step(equil_steps)

    samples = []
    completed = 0
    status = "ok"
    notes = ""
    while completed < prod_steps:
        chunk = min(sample_interval, prod_steps - completed)
        simulation.step(chunk)
        completed += chunk
        state = simulation.context.getState(getEnergy=True)
        K = state.getKineticEnergy().value_in_unit(unit.kilojoule_per_mole)
        U = state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
        T_inst = 2.0 * K / (max(1, 3 * N - 3) * epsilon)
        positions = get_positions(simulation)
        P = lj_virial_pressure(positions, L, T_inst, rcut, sigma, epsilon)
        samples.append({"T": T_inst, "P": P, "U": float(U), "K": float(K), "E": float(U + K)})
        if not math.isfinite(T_inst) or T_inst > 10.0 * T:
            status = "unstable_temperature"
            notes = "temperature became non-finite or too high"
            break

    final_positions = get_positions(simulation)
    t_mean, _, _ = sample_stats([row["T"] for row in samples])
    p_mean, p_std, p_sem = sample_stats([row["P"] for row in samples])
    u_mean, u_std, u_sem = sample_stats([row["U"] for row in samples])
    k_mean, _, _ = sample_stats([row["K"] for row in samples])
    e_mean, _, _ = sample_stats([row["E"] for row in samples])
    point = {
        "run_id": run_id, "T_target": float(T), "T_mean": t_mean,
        "rho_target": float(rho), "rho_mean": N / (L ** 3), "N": N, "L": L,
        "dt": dt, "equil_steps": equil_steps, "prod_steps": completed, "seed": int(seed),
        "P_mean": p_mean, "P_std": p_std, "P_sem": p_sem,
        "U_mean": u_mean, "U_std": u_std, "U_sem": u_sem,
        "K_mean": k_mean, "E_mean": e_mean, "status": status, "notes": notes,
    }
    profiles = compute_z_counts(final_positions, L, int(params["profile_bins"]), run_id)
    return point, profiles


## Тестовый короткий прогон

In [ ]:
test_point, test_profile = run_eos_point(
    PARAMS,
    PARAMS["temperatures"][0],
    PARAMS["densities"][0],
    PARAMS["seeds"][0],
    "test",
)
test_point


## Расчёт EOS-сетки

In [ ]:
points = []
profiles = []
tasks = [(T, rho, seed) for T in PARAMS["temperatures"] for rho in PARAMS["densities"] for seed in PARAMS["seeds"]]

for idx, (T, rho, seed) in enumerate(tqdm(tasks), start=1):
    run_id = f"eos_{idx:04d}"
    point, profile = run_eos_point(PARAMS, T, rho, seed, run_id)
    points.append(point)
    profiles.extend(profile)

points_df = pd.DataFrame(points, columns=EOS_FIELDS)
profiles_df = pd.DataFrame(profiles, columns=PROFILE_FIELDS)
points_df


## Сохранение таблиц

In [ ]:
points_df.to_csv(DATA_DIR / "eos_points.csv", index=False)
profiles_df.to_csv(DATA_DIR / "eos_final_profiles.csv", index=False)
print(DATA_DIR / "eos_points.csv")
print(DATA_DIR / "eos_final_profiles.csv")
print("EOS rows:", len(points_df), "profile rows:", len(profiles_df))


## Fit уравнения Ван-дер-Ваальса

In [ ]:
def vdw_pressure(rho, T, a, b):
    return rho * T / (1.0 - b * rho) - a * rho * rho

def fit_vdw(points_df):
    fit_df = points_df[(points_df["status"] == "ok") & np.isfinite(points_df["rho_mean"]) & np.isfinite(points_df["T_mean"]) & np.isfinite(points_df["P_mean"])].copy()
    if len(fit_df) < 3:
        raise ValueError("Need at least 3 EOS points for VdW fit")
    data = list(zip(fit_df["rho_mean"].astype(float), fit_df["T_mean"].astype(float), fit_df["P_mean"].astype(float)))

    def best_a_for_b(b):
        numerator = 0.0
        denominator = 0.0
        for rho, T, P in data:
            base = rho * T / (1.0 - b * rho)
            x = rho * rho
            numerator += x * (base - P)
            denominator += x * x
        return max(0.0, numerator / denominator) if denominator else 0.0

    def objective(b):
        a = best_a_for_b(b)
        return sum((P - vdw_pressure(rho, T, a, b)) ** 2 for rho, T, P in data)

    upper = 0.95 / max(rho for rho, _, _ in data)
    result = minimize_scalar(objective, bounds=(0.0, upper), method="bounded")
    b = float(result.x)
    a = float(best_a_for_b(b))
    residuals = []
    for rho, T, P in data:
        P_fit = float(vdw_pressure(rho, T, a, b))
        residuals.append({"rho": rho, "T": T, "P_md": P, "P_vdw": P_fit, "residual": P - P_fit})
    return {"a": a, "b": b, "n_fit_points": len(data), "fit_region": {"status": ["ok"], "source": "all finite ok rows"}, "residuals": residuals}

vdw_fit = fit_vdw(points_df)
(DATA_DIR / "vdw_fit.json").write_text(json.dumps(vdw_fit, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
vdw_fit


## Графики

In [ ]:
def plot_isotherms(points_df, path):
    ok = points_df[points_df["status"] == "ok"].copy()
    plt.figure(figsize=(6, 4))
    for T in sorted(ok["T_target"].unique()):
        sub = ok[ok["T_target"] == T].sort_values("rho_mean")
        plt.plot(sub["rho_mean"], sub["P_mean"], marker="o", label=f"T={T:g}")
    plt.xlabel("rho")
    plt.ylabel("P")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.show()

def plot_vdw_fit(points_df, fit, path):
    ok = points_df[points_df["status"] == "ok"].copy()
    rho_grid = np.linspace(max(1e-9, 0.9 * ok["rho_mean"].min()), 1.05 * ok["rho_mean"].max(), 200)
    plt.figure(figsize=(7, 4.5))
    for T in sorted(ok["T_target"].unique()):
        sub = ok[ok["T_target"] == T].sort_values("rho_mean")
        T_curve = float(sub["T_mean"].mean())
        plt.plot(sub["rho_mean"], sub["P_mean"], marker="o", linestyle="none", label=f"MD T={T:g}")
        plt.plot(rho_grid, vdw_pressure(rho_grid, T_curve, fit["a"], fit["b"]), label=f"VdW T={T:g}")
    plt.xlabel("rho")
    plt.ylabel("P")
    plt.legend(fontsize="small", ncol=2)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.show()

def plot_residuals(fit, path):
    residuals = pd.DataFrame(fit["residuals"])
    plt.figure(figsize=(6, 4))
    for T in sorted(residuals["T"].unique()):
        sub = residuals[residuals["T"] == T].sort_values("rho")
        plt.plot(sub["rho"], sub["residual"], marker="o", label=f"T={T:.3g}")
    plt.axhline(0.0, color="0.7", linewidth=1)
    plt.xlabel("rho")
    plt.ylabel("P_MD - P_VdW")
    plt.legend(fontsize="small")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.show()

plot_isotherms(points_df, FIGURES_DIR / "eos_isotherms.png")
plot_vdw_fit(points_df, vdw_fit, FIGURES_DIR / "vdw_fit.png")
plot_residuals(vdw_fit, FIGURES_DIR / "vdw_residuals.png")


## Архивирование результатов

In [ ]:
import zipfile

zip_path = Path("lj_vdw_results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in (DATA_DIR, FIGURES_DIR):
        for path in folder.rglob("*"):
            if path.is_file():
                zf.write(path, path.as_posix())
zip_path
